# jaccode: train our own Jac model, free

Everything runs on Colab's free GPU. You need no API key and pay nothing.
1. A free open teacher model (Qwen3.5-9B) writes Jac solutions inside Colab.
2. The real Jac compiler and tests check every answer, and only passing ones are kept.
3. Qwen3.5-2B is fine-tuned on that data.
4. The trained model is scored against the untrained one on the Jac benchmark.
5. Optional round 2: the trained model solves more problems itself, and it trains again on its verified answers.

**Before you start:** Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**.
Progress is saved to your Google Drive, so if Colab disconnects, run all again and it carries on.
The defaults take roughly 3–4 hours. That's an estimate: this notebook hasn't been run end to end yet.

In [ ]:
PROBLEMS = 200        # Python problems the teacher solves in Jac (more = smarter model, longer run)
AGENT_TASKS = 40      # multi-step agent tasks recorded with the teacher
ROUND2 = True         # let the trained model solve more problems, then retrain on its verified answers
TEACHER = "hf.co/unsloth/Qwen3.5-9B-GGUF:Q4_K_M"   # free teacher (Apache 2.0)
BASE = "unsloth/Qwen3.5-2B"                          # model we train
BASE_GGUF = "hf.co/unsloth/Qwen3.5-2B-GGUF:Q5_K_M"   # same model untrained, for the comparison
REPO = "https://github.com/hernandeztien51-crypto/Jaccode"
USE_DRIVE = True      # keep data and checkpoints in Google Drive so a disconnect doesn't lose work

import os, subprocess, time
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]
WORK = "/content/work"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/jaccode-training"
os.makedirs(f"{WORK}/data", exist_ok=True); os.makedirs(f"{WORK}/out", exist_ok=True)

def ollama_up():
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)

def ollama_down():  # frees the GPU for training
    subprocess.run(["pkill", "-f", "ollama serve"]); time.sleep(3)

## 1. Install tools

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!curl -fsSL https://raw.githubusercontent.com/jaseci-labs/jaseci/main/scripts/install.sh | bash
!jac --version
!curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null && apt-get install -y -qq nodejs > /dev/null
!node --version
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
!pip install -q --upgrade unsloth unsloth_zoo

## 2. Get the code and check the benchmark against the real compiler

In [ ]:
%cd /content
!rm -rf jaccode jaseci
!git clone -q --depth 1 {REPO} jaccode
!git clone -q --depth 1 https://github.com/jaseci-labs/jaseci jaseci
%cd /content/jaccode
!npm install --no-audit --no-fund --silent && npm run build --silent
!pip install -q -r training/requirements.txt
# Training data and checkpoints live in WORK (Drive), linked into the repo.
!rm -rf training/data training/out && ln -s {WORK}/data training/data && ln -s {WORK}/out training/out
!node dist/cli.js eval --reference --quiet --out {WORK}/reference.json

In [ ]:
import json
ref = json.load(open(f"{WORK}/reference.json"))["results"]
VALID = [r["id"] for r in ref if r["passed"]]
ONLY = ",".join(VALID)
print(f"{len(VALID)}/{len(ref)} benchmark tasks verified by the real compiler")
for r in ref:
    if not r["passed"]:
        print(f"--- needs fixing: {r['id']}\n{(r.get('checkOutput') or '')[-600:]}\n{(r.get('testOutput') or '')[-600:]}")

## 3. Build verified training data with the free teacher

In [ ]:
%cd /content/jaccode/training
ollama_up()
!ollama pull {TEACHER}
!python build_corpus.py --jaseci /content/jaseci --out data/corpus.jsonl --verify
!python translate.py --mbpp --limit {PROBLEMS} --teacher ollama:{TEACHER} --out data/translated.jsonl
!python make_agent_tasks.py --translated data/translated.jsonl --out data/agent_tasks
!node ../dist/cli.js eval --tasks data/agent_tasks --limit {AGENT_TASKS} -m ollama:{TEACHER} --record data/runs.jsonl --quiet
!python prepare_sft.py --translated data/translated.jsonl --runs data/runs.jsonl --corpus data/corpus.jsonl \
    --general HuggingFaceTB/smoltalk --general-config all --out data/sft

## 4. Fine-tune

In [ ]:
ollama_down()
!python train_sft.py --data data/sft --out out/jac-2b --base {BASE} --quants q5_k_m

## 5. Score it: untrained vs trained

In [ ]:
import glob, pathlib
def register(name, gguf_dir):
    gguf = sorted(glob.glob(f"{gguf_dir}/*.gguf"), key=lambda p: pathlib.Path(p).stat().st_size)[-1]
    mf = pathlib.Path(gguf_dir) / "Modelfile"   # Unsloth writes one with the chat template
    extra = [l for l in (mf.read_text().splitlines() if mf.exists() else []) if not l.startswith("FROM")]
    pathlib.Path("Modelfile").write_text("\n".join([f"FROM {gguf}", *extra]) + "\n")
    subprocess.run(["ollama", "create", name, "-f", "Modelfile"], check=True)
    return gguf

ollama_up()
GGUF = register("jaccode-jac", "out/jac-2b/gguf")
!ollama pull {BASE_GGUF}
!node ../dist/cli.js eval -m ollama:{BASE_GGUF} --only {ONLY} --out {WORK}/base.json --quiet
!node ../dist/cli.js eval -m ollama:jaccode-jac --only {ONLY} --out {WORK}/trained.json --quiet
!python compare.py {WORK}/base.json {WORK}/trained.json

## 6. Round 2 (optional): the model teaches itself

In [ ]:
if ROUND2:
    # The trained model attempts more problems; the compiler keeps only its correct answers.
    !python translate.py --mbpp --limit {PROBLEMS * 2} --teacher ollama:jaccode-jac --out data/translated.jsonl --samples 4
    !python prepare_sft.py --translated data/translated.jsonl --runs data/runs.jsonl --corpus data/corpus.jsonl \
        --general HuggingFaceTB/smoltalk --general-config all --out data/sft
    ollama_down()
    !python train_sft.py --data data/sft --out out/jac-2b-r2 --base {BASE} --quants q5_k_m
    ollama_up()
    GGUF = register("jaccode-jac-r2", "out/jac-2b-r2/gguf")
    !node ../dist/cli.js eval -m ollama:jaccode-jac-r2 --only {ONLY} --out {WORK}/trained_r2.json --quiet
    !python compare.py {WORK}/base.json {WORK}/trained.json {WORK}/trained_r2.json

## 7. Keep the model

The model file is in your Drive under `jaccode-training/out/`. To use it on a computer, with no API key:
```
ollama create jaccode-jac -f Modelfile      # a Modelfile containing: FROM ./your-model.gguf
jaccode -m ollama:jaccode-jac
```
To get a smarter model, raise `PROBLEMS` and `AGENT_TASKS` and run again. Keep whichever version scores best.

In [ ]:
print("Model file:", GGUF)
from google.colab import files
files.download(GGUF)